In [2]:
%pip install matplotlib seaborn scikit-learn vaderSentiment

  Using cached matplotlib-3.10.9-cp311-cp311-win_amd64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp311-cp311-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp311-cp311-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp311-cp311-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
Using cached matplotlib-3.10.9-cp311-cp311-win_amd64.whl (8.2 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)
Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl (225 kB)
Using cached cycler-0.12.1-py3-none-an


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import classification_report, confusion_matrix


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
analyzer = SentimentIntensityAnalyzer()



Now let us load the data

In [2]:
df = pd.read_csv("../data/clean/all_apps_clean.csv")
print(df.shape)
df[["app_name", "rating", "review_clean"]].head(3)

(10386, 16)


,app_name,rating,review_clean
0,Cash App,1,keeps popping up as spam just horrible setup i have a samsung flip the cashapp keep popping up anytime i open use anything on my phone and a few times it just opens while im listening to music talking on the phone the app interrupted twice while completing this summary
1,Cash App,1,horrible app one day its totally working fine then they the next they deactivate you account and cant restart it but they cant tell you why even the appeal it is worthless you get no where
2,Cash App,5,this ap has served me so well


NOW WE BRING ON THE VADER SCORING FUNCTION

Now we calculate the sentiment. We aren't just getting one number; we are creating a systematic way to process every review

In [3]:
def get_vader_score(text):
    if pd.isna(text):
        return 0
    return analyzer.polarity_scores(str(text))["compound"]

#now we apply scoring
df['vader_compound'] = df['review_clean'].apply(get_vader_score)

#now we preview the scores
df[['review_clean', 'rating', 'vader_compound']].sort_values('vader_compound').head(5)


,review_clean,rating,vader_compound
7925,it works for its stated purpose at least but is leading the vanguard in the enshittification this app is continuously getting worse about shoving offensively bad ads in your face forcing notifications nagging and an utterly useless pseudosupport chatbot that only serves as a wall between the customer and an actually usable product i would replace it if it werent also close to a monopoly which doubles the resentment even more i always regret launching this vapid hole of marketing hell,2,-0.9907
3936,scams abound on this platform and they know about it and make it impossible to report suspicious activity the customer service page is a joke and chat bot keeps directing you back to itself to report but doesnt do anything i keep trying to report a fraudulent payment sent to me by a stranger unsolicited i keep getting these stupid scam payments and there seems to be no way to stop them my account is for business and it is even hidden in search how can i make it stop,1,-0.9774
5396,do not use chime period nothing but serious headaches and problems false advertising about my pay its not when you want it its when they decide to give it to you and how to give it to you and how much they want to give you it will have you so screwed up and being desperate for money this bank has the worst and i mean worst customer service ive ever seen they do not understand english and cant ever help you solve any issues they suckperiod nothing but lies,1,-0.9759
6487,completely trash use to be great but now they are doing shady things they offer a mypay which will loan you money until pay day but these fools have now made it to where they suck any money that goes into your account before you get paid just sucked out dollars that was put in to cover a bill before i even logged in they had sucked it out and made me need to borrow again for a fee or wait hours you people suck im done with you and any dev responses can suck and fat egg,1,-0.9747
6914,dont use this banking app i get charged fraudulent charges a couple of times a month i dispute the charges and then they deny my dispute but im pretty sure they dont investigate just deny i know for a fact the charges are fraudulent and they still deny them recently i had a fraudulent charge i sent them all the information and the proof that it was fraudulent and got ghosted i guarantee you it will still be denied very unprofessional and uncaring will help website steal fromu,1,-0.9743


What we are doing

In this step, we are generating a SENTIMENT BASELINE using VADER - a lexicon and rule-based sentiment analysis tool specifically attuned to sentiments expressed in social media and product reviews.


For every review in the dataset, we calculate a compound score. This score is a metric that calculates the sum of all the lexicon ratings which have been normalized between -1 (extreme negative) and +1 (extreme positive)



The Business Hypothesis

We aren't using VADER because we expect it to be 100% accurate. We are using it to test a primary hypothesis:

Fintech users often communicate critical product failures using technical, non-emotional language that simple sentiment tools (like VADER) will misclassify as 'Neutral'



By sorting the results by the lowest compound scores, we can immediately verify if VADER’s lexicon "hits" the obvious angry reviews. If the top 5 negative scores show clear user frustration, VADER is functioning correctly as a baseline. The real value, however, will be found in the mismatches where the user left 1 star but VADER gave a score near 0.